In [1]:
# Load checkpoint, tokenizer và chuẩn bị tập validation và test
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import os
checkpoint_path = '../models/vit5-base-bs64-ml128'
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Hàm load dữ liệu val/test (giả sử đã có sẵn file json)
import json
def load_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

val_data = load_dataset('../data/val_dataset_full.json')
test_data = load_dataset('../data/test_dataset_full.json')

/home/vinh/anaconda3/envs/vit5-chatbot/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-24 10:51:03.734616: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-24 10:51:03.748722: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753329063.762784 2936053 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753329063.767050 2936053 cuda_blas.c

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
from tqdm import tqdm
from bert_score import score as bert_score
from rouge_score import rouge_scorer
import sacrebleu

def evaluate(model, tokenizer, data, device, max_gen_len=128, max_samples=None, max_input_len=512, batch_size=8):
    references = []
    predictions = []
    
    # Giới hạn số lượng mẫu nếu max_samples được chỉ định
    eval_data = data[:max_samples] if max_samples else data

    # Xử lý theo batch
    for i in tqdm(range(0, len(eval_data), batch_size), desc="Evaluating"):
        batch_data = eval_data[i:i+batch_size]
        
        # Chuẩn bị batch input
        input_texts = [item["prompt"] for item in batch_data]
        ref_outputs = [item["response"] for item in batch_data]
        
        # Tokenize batch
        inputs = tokenizer(
            input_texts, 
            return_tensors="pt", 
            truncation=True, 
            padding=True, 
            max_length=max_input_len
        ).to(device)
        
        # Generate batch predictions
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_length=max_gen_len,
                num_beams=1,  # Giảm beam search để tăng tốc
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
        
        # Decode batch predictions
        batch_predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        # Thêm vào lists
        predictions.extend(batch_predictions)
        references.extend(ref_outputs)

    # 1. BLEU (sử dụng sacreBLEU)
    bleu = sacrebleu.corpus_bleu(predictions, [references]).score

    # 2. ROUGE-L
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l_scores = [rouge.score(ref, pred)['rougeL'].fmeasure for ref, pred in zip(references, predictions)]
    avg_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)

    P, R, F1 = bert_score(predictions, references, lang="vi", model_type="bert-base-multilingual-cased", verbose=True)
    avg_bertscore_f1 = F1.mean().item()
    # 3. BERTScore (sử dụng mô hình tiếng Việt)
    # try:
    #     # Thử sử dụng PhoBERT
    #     P, R, F1 = bert_score(predictions, references, lang="vi", model_type="distilbert/distilbert-base-uncased", verbose=True)
    #     avg_bertscore_f1 = F1.mean().item()
    # except:
    #     try:
    #         # Nếu không có PhoBERT, sử dụng multilingual BERT
    #         P, R, F1 = bert_score(predictions, references, lang="vi", model_type="bert-base-multilingual-cased", verbose=True)
    #         avg_bertscore_f1 = F1.mean().item()
    #     except:
    #         # Fallback về model mặc định
    #         P, R, F1 = bert_score(predictions, references, lang="vi", verbose=True)
    #         avg_bertscore_f1 = F1.mean().item()

    return {
        "BLEU": bleu,
        "ROUGE-L": avg_rouge_l,
        "BERTScore-F1": avg_bertscore_f1
    }

# Chạy đánh giá với 10 mẫu dữ liệu đầu tiên và batch processing
# Sử dụng batch_size=4 để xử lý nhiều mẫu cùng lúc
print(f"Đánh giá với {len(val_data)} mẫu dữ liệu đầu tiên (batch processing)...")
val_metrics = evaluate(model, tokenizer, val_data, device, 
                      max_gen_len=128, max_input_len=512, batch_size=256)
test_metrics = evaluate(model, tokenizer, test_data, device, 
                       max_gen_len=128, max_input_len=512, batch_size=256)

print("Validation Set Metrics:")
print(val_metrics)

print("Test Set Metrics:")
print(test_metrics)


Đánh giá với 4565 mẫu dữ liệu đầu tiên (batch processing)...


Evaluating: 100%|██████████| 18/18 [00:54<00:00,  3.00s/it]


calculating scores...
computing bert embedding.


100%|██████████| 143/143 [00:10<00:00, 13.55it/s]


computing greedy matching.


...

calculating scores...
computing bert embedding.


100%|██████████| 143/143 [00:11<00:00, 12.73it/s]


computing greedy matching.


100%|██████████| 72/72 [00:00<00:00, 86.98it/s]


done in 12.10 seconds, 377.34 sentences/sec
Validation Set Metrics:
{'BLEU': 24.84973653106292, 'ROUGE-L': 0.5282683530856825, 'BERTScore-F1': 0.8354191780090332}
Test Set Metrics:
{'BLEU': 24.754373327840504, 'ROUGE-L': 0.5296402667958118, 'BERTScore-F1': 0.836317777633667}
